In [1]:
# ЯЧЕЙКА 1
# Импорты. Только необходимое для быстрого эксперимента.

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

from catboost import CatBoostClassifier


In [5]:
# ЯЧЕЙКА 2
# Загрузка sample-датасета с таргетом.
# Здесь мы больше НЕ ТРОГАЕМ пайплайн.

DATASET_PATH = "../data/processed/dataset_with_target_sample.pkl"

df = pd.read_pickle(DATASET_PATH)

df.shape, df.head()


((300000, 75),
         id  pre_since_opened_max  pre_since_confirmed_max  pre_pterm_max  \
 0   404846                    12                       10             16   
 1  2580313                    15                       14             11   
 2  2552086                    17                        9             16   
 3   370876                    13                       16             17   
 4   239330                    17                       17             17   
 
    pre_fterm_max  pre_till_pclose_max  pre_till_fclose_max  \
 0             11                   15                   14   
 1             16                   16                   13   
 2              8                   14                   11   
 3             15                   12                   15   
 4              9                   12                   11   
 
    pre_loans_credit_limit_max  pre_loans_next_pay_summ_max  \
 0                          17                            6   
 1             

In [6]:
# ЯЧЕЙКА 2.1
df.filter(like="enc_loans").head()


""
0
1
2
3
4


In [7]:
# ЯЧЕЙКА 3
# Базовые проверки целостности датасета.

assert "flag" in df.columns, "Нет таргета flag"
assert "id" in df.columns, "Нет id"

df["flag"].value_counts(normalize=True)


flag
0    0.967097
1    0.032903
Name: proportion, dtype: float64

In [8]:
# ЯЧЕЙКА 4
# Разделение на признаки и таргет.
# id исключаем — он не признак.

X = df.drop(columns=["flag", "id"])
y = df["flag"]

X.shape, y.value_counts(normalize=True)


((300000, 73),
 flag
 0    0.967097
 1    0.032903
 Name: proportion, dtype: float64)

In [9]:
# ЯЧЕЙКА 5
# Train / Validation split.
# Стратификация обязательна.

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

X_train.shape, X_val.shape


((240000, 73), (60000, 73))

In [10]:
# ЯЧЕЙКА 7.1
# Баланс классов.

pos = y_train.sum()
neg = len(y_train) - pos
scale_pos_weight = neg / pos

scale_pos_weight


29.39128783082183

In [11]:
# ЯЧЕЙКА 6
# Обучение CatBoost.
# Параметры умеренные — цель быстро понять уровень AUC.

model = CatBoostClassifier(
    iterations=1200,
    learning_rate=0.04,
    depth=8,
    loss_function="Logloss",
    eval_metric="AUC",
    scale_pos_weight=scale_pos_weight,
    random_seed=42,
    verbose=100,
)

model.fit(
    X_train,
    y_train,
    eval_set=(X_val, y_val),
    use_best_model=True,
)


0:	test: 0.6403080	best: 0.6403080 (0)	total: 183ms	remaining: 3m 39s
100:	test: 0.6659531	best: 0.6659956 (99)	total: 5.54s	remaining: 1m
200:	test: 0.6689608	best: 0.6690726 (196)	total: 11.4s	remaining: 56.5s
300:	test: 0.6694061	best: 0.6694761 (298)	total: 17.5s	remaining: 52.3s
400:	test: 0.6668765	best: 0.6694761 (298)	total: 24.5s	remaining: 48.9s
500:	test: 0.6623840	best: 0.6694761 (298)	total: 31.9s	remaining: 44.5s
600:	test: 0.6584831	best: 0.6694761 (298)	total: 39.4s	remaining: 39.3s
700:	test: 0.6550865	best: 0.6694761 (298)	total: 48.1s	remaining: 34.2s
800:	test: 0.6522007	best: 0.6694761 (298)	total: 57.5s	remaining: 28.7s


KeyboardInterrupt: 

In [34]:
# ЯЧЕЙКА 7
# Расчёт ROC-AUC на валидации.

proba_val = model.predict_proba(X_val)[:, 1]
roc_auc = roc_auc_score(y_val, proba_val)

roc_auc


0.6980532361711451

In [22]:
# ЯЧЕЙКА 8
# Быстрая интерпретация важности признаков.
# Смотрим, есть ли enc_paym_*_max в топе.

feat_imp = (
    pd.DataFrame({
        "feature": X.columns,
        "importance": model.get_feature_importance()
    })
    .sort_values("importance", ascending=False)
)

feat_imp.head(20)


,feature,importance
205,enc_loans_credit_type_0,3.374628
203,enc_loans_credit_type_mean,2.189914
33,pre_loans_outstanding_mean_max,2.142298
73,is_zero_loans530_mean_max,2.083138
207,enc_loans_credit_type_2,2.064238
72,is_zero_loans530_mean_mean,1.936173
89,pre_util_mean_max,1.826738
47,pre_loans_credit_cost_rate_max_max,1.785954
100,is_zero_util_mean_mean,1.656970
210,enc_loans_credit_type_5,1.624469
